In [ ]:
import datetime
import textwrap
from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.dates import DateFormatter, MonthLocator
from statsmodels.graphics.tsaplots import plot_acf

In [ ]:
df = pd.read_csv("../data/turingAI_forecasting_challenge_dataset.csv")

## Data cleaning

Handling typos.

In [ ]:
df.loc[(df.coverage_label == "SWAST") & (df.coverage == "SWAST"), ["coverage_label", "coverage"]] = "SWASFT"
df.loc[df.metric_name == "Ambulance Handovers 15mins (Last Hour)", "metric_name"] = (
    "Ambulance Handovers 15 mins (Last Hour)"
)
df.loc[df.metric_name == "DtA P1 slots booked", "metric_name"] = "DtA P1 Slots Booked"
df.loc[df.metric_name == "DtA P2 referrals received", "metric_name"] = "DtA P2 Referrals Received"
df.loc[df.metric_name == "DtA P3 beds occupied", "metric_name"] = "DtA P3 Beds Occupied"

Forecasts for days D+1 to D+10 can only use data available up to midday on day D. Therefore, we create a new column, midday_day. The observation frequency varies across metrics, from daily to every 15 minutes. Data received before or by 12:00 pm are attributed to the same day; data received after 12:00 pm are attributed to the next day. Features are then averaged over midday_day.

In [ ]:
# Date formatting
df["dt"] = pd.to_datetime(df["dt"], format="mixed", utc=True)
# "mixed", to infer the format for each element individually.
# This is risky, and you should probably use it along with dayfirst.

# Filter out assessment dataset
cutoff = pd.Timestamp("2025-09-30 00:00:00", tz="UTC")
df = df.loc[df["dt"] <= cutoff]

# Data cleaning: midday threshold
df["midday_day"] = np.where(
    df["dt"].dt.time <= datetime.time(hour=12),
    df["dt"].dt.date,
    (df["dt"] + pd.Timedelta(days=1)).dt.date,
)

## Exploratory Data Analysis EDA

### target variable

Analysing the target variable, estimated avoidable deaths, starting with the time series plot and the autocorrelation function.

In [ ]:
outcome_df = df.loc[(df.variable_type == "outcome"), ["dt", "value"]]
outcome_df = outcome_df.rename(columns={"dt": "day", "value": "estimated_avoidable_deaths"})

In [ ]:
fig, ax = plt.subplots()
sns.lineplot(data=outcome_df, x="day", y="estimated_avoidable_deaths")
ax.xaxis.set_major_locator(MonthLocator(interval=3))
ax.xaxis.set_major_formatter(DateFormatter("%b '%y"))
ax.tick_params(axis="x", labelrotation=45)
ax.set_title("Estimated avoidable deaths")
ax.grid(axis="x")

In [ ]:
fig, ax = plt.subplots()
plot_acf(outcome_df["estimated_avoidable_deaths"], lags=49, ax=ax)
ax.set_xticks(range(0, 50, 7))
ax.set_xlabel("lag")
ax.set_ylabel("autocorrelation")
ax.grid(True, axis="x")

The target variable exhibits weekly (day-of-week) seasonality. The autocorrelation function (ACF) peaks at lags of 7, 14, etc. days. The ACF is defined as the correlation between a time series and its lagged versions for lags 0, ..., N.

Here, we plot the distribution of estimated avoidable deaths and its variance, shown as a 30-day rolling variance. There are periods where the variance exhibits level shifts. This non-stationarity is currently addressed by applying a log1p transformation to the target variable.

In [ ]:
fig, ax = plt.subplots()
sns.histplot(outcome_df, x="estimated_avoidable_deaths", bins=30, fill=True, log_scale=False)
ax.set_title("Estimated avoidable deaths")

In [ ]:
outcome_df["rolling_var"] = (outcome_df["estimated_avoidable_deaths"]).rolling(window=30, min_periods=7).var()

fig, ax = plt.subplots()
sns.lineplot(data=outcome_df, x="day", y="rolling_var")
ax.xaxis.set_major_locator(MonthLocator(interval=3))
ax.xaxis.set_major_formatter(DateFormatter("%b '%y"))
ax.tick_params(axis="x", labelrotation=45)
ax.set_title("30-day rolling variance")
ax.grid(axis="x")

Plotting box plots of the target variable grouped by day of week and by month to show both seasonalities.

In [ ]:
outcome_df["dayofweek"] = outcome_df["day"].dt.dayofweek
# The day of the week with Monday=0, Sunday=6.

fig, ax = plt.subplots()
sns.boxplot(data=outcome_df, x="dayofweek", y="estimated_avoidable_deaths", ax=ax)
# df.boxplot(column="estimated_avoidable_deaths", by="dayofweek", ax=ax)
ax.set_title("Weekly seasonality")

In [ ]:
outcome_df["month"] = outcome_df["day"].dt.month

fig, ax = plt.subplots()
sns.boxplot(data=outcome_df, x="month", y="estimated_avoidable_deaths", ax=ax)
# df.boxplot(column="estimated_avoidable_deaths", by="month", ax=ax)
ax.set_title("Monthly seasonality")

The target shows clear weekly and monthly seasonality, as well as autocorrelation at 7-day intervals and non-stationarity in variance.

To capture these patterns, we will explicitly engineer time-based and lag features:

- Day-of-week averages (weekly seasonality)
- Month-of-year averages (annual seasonality)
- Lagged values (e.g. 7-day lag) to capture autocorrelation

### feature availability hierarchy

Aggregating the data by averaging over day, feature, and coverage label, then plotting a heatmap to show feature sparsity. Each coverage label has only a small subset of non-null features from the ~220 available.


In [ ]:
long_df = (
    df[["midday_day", "metric_name", "coverage_label", "value"]]
    .groupby(["midday_day", "metric_name", "coverage_label"], as_index=False)["value"]
    .mean()
    .sort_values(by=["metric_name", "midday_day"], ignore_index=True)
)

In [ ]:
hierarchy_df = long_df.groupby(["coverage_label", "metric_name"])["value"].count().unstack(fill_value=0)
hierarchy_binary = (hierarchy_df > 0).astype(int)

fig, ax = plt.subplots(figsize=(20, 10))
sns.heatmap(hierarchy_binary, cbar=False, cmap="YlGnBu", xticklabels=False, ax=ax)
ax.set_ylabel("Coverage label", fontsize=18)
ax.set_xlabel("Metric names", fontsize=18)
ax.tick_params(axis="y", labelsize=12)
ax.set_title("Feature availability hierarchy", fontsize=24)

### missing values

Handling missing values. In most cases, missingness occurs because data for certain features started being recorded after 2023-03-16. In other cases, there are occasional missing values. In the latter case, we forward fill.

In [ ]:
# overwrite
long_df = (
    df[["midday_day", "metric_name", "value"]]
    .groupby(["midday_day", "metric_name"], as_index=False)["value"]
    .mean()
    .sort_values(by=["metric_name", "midday_day"], ignore_index=True)
)

long_df["midday_day"] = pd.to_datetime(long_df["midday_day"])

Transforming the data into wide format by pivoting so that each column represents a feature.

In [ ]:
wide_df = long_df.pivot_table(
    index=["midday_day"],
    columns=["metric_name"],
    values="value",
)
wide_df.columns = wide_df.columns.str.lower().str.replace(" ", "_")

Creating a summary table with the following columns for each feature:
- first day the feature appears in the dataset
- last day the feature appears in the dataset
- total number of missing values (days without data)
- maximum number of consecutive days with missing values
- head_missing: number of days after 2023-03-16 before the feature starts being recorded
- else_missing: number of missing values between the first and last day the feature is recorded

Better analysed in Google Sheets (summary.to_clipboard()). In the vast majority of cases, missingness arises because features start being recorded days or months after the initial date (2023-03-16).

Based on this, we can either drop features with high missingness or start training from a later date when more features are available. We should also check how strong the correlation of these features is with the target variable.

In [ ]:
# 1. Ensure chronological order (critical for consecutive counts)
df = wide_df.sort_index()

# 2. Boolean mask of the entire 2D DataFrame
is_missing = df.isna()

# 3. The 2D Vectorized Trick for Max Consecutive Missing
# Calculate a running total of missing values down every column
missing_cumsum = is_missing.cumsum()

# Create a "reset" baseline: grab the cumsum number wherever a valid value exists,
# and carry that number forward through the missing blocks. Fill leading NaNs with 0.
reset_blocks = missing_cumsum.where(~is_missing).ffill().fillna(0)

# Subtracting the baseline from the running total gives the consecutive count!
consecutive_missing = missing_cumsum - reset_blocks
max_consecutive = consecutive_missing.max().astype(int)

# 4. Build the final summary DataFrame
summary = pd.DataFrame(
    {
        # pandas has native methods to find the first/last valid index of a Series
        "first_midday_day": df.apply(lambda col: col.first_valid_index()),
        "last_midday_day": df.apply(lambda col: col.last_valid_index()),
        "total_missing": is_missing.sum().astype(int),
        "max_consecutive_missing": max_consecutive,
    }
)

# Move the metric names from the index back to a standard column
summary = summary.rename_axis("metric_name").reset_index()

summary["head_missing"] = (summary["first_midday_day"] - pd.to_datetime("2023-03-16")).dt.days
summary["else_missing"] = summary["max_consecutive_missing"] - summary["head_missing"]
summary = summary.sort_values(
    by=["total_missing", "max_consecutive_missing"], ascending=[False, False], ignore_index=True
)

summary.head()

In [ ]:
# forward fill
wide_df = wide_df.ffill()

### correlation analysis

In [ ]:
correlations = (
    wide_df.corrwith(wide_df["estimated_avoidable_deaths"])
    .drop("estimated_avoidable_deaths")
    .sort_values(ascending=False)
)

In [ ]:
# Features with the highest proportion of missing values (first_midday_day = 2024-05-28)
cols = [
    "bed_occupancy_adult",
    "bed_occupancy_picu",
    "intensive_caseload",
    "oot_internal_placements",
    "urgent_referrals",
]

In [ ]:
correlations[cols]

Plotting a heatmap of the correlation matrix for the top 10 features most correlated with the target variable.

In [ ]:
target = "estimated_avoidable_deaths"

# Since the target's correlation with itself is 1.0, it naturally stays at the top)
top_cols = wide_df.corrwith(wide_df[target]).sort_values(ascending=False).head(11).index

corr = wide_df[top_cols].corr(min_periods=5)

# Wrap long labels for readability
wrapped_labels = [textwrap.fill(str(col), 35) for col in corr.columns]
corr.index = corr.columns = wrapped_labels

plt.figure(figsize=(11, 9), dpi=200)

# Create a mask to hide the upper triangle and main diagonal
mask = np.triu(np.ones_like(corr, dtype=bool))

# Draw the heatmap
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 6, "alpha": 0.8},
    center=0,
    vmin=-1,
    vmax=1,
    cmap="RdBu_r",
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.75, "aspect": 30, "label": "Pearson Correlation"},
)

plt.title("Top 10 Positive Correlations — Sorted by Estimated Avoidable Deaths", pad=20, fontsize=12, fontweight="bold")

plt.xticks(rotation=45, ha="right", fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tick_params(left=False, bottom=False)

plt.tight_layout()
plt.show()

### cross-correlation

Plotting cross-correlation with the target variable for the top 15 features (by correlation at lag 0). Some features exhibit clear weekly seasonality, with correlation decreasing at lags 1–3 and peaking again at lag 7, and so on. This suggests that STL decomposition may be worth exploring.

In [ ]:
target = "estimated_avoidable_deaths"
max_lag = 14

# 1. Get the top 15 features positively correlated with the target at Lag 0
top_15 = wide_df.corrwith(wide_df[target]).drop(target).sort_values(ascending=False).head(15).index

# 2. Calculate cross-correlations for Lags 0 to 14
# We use a dictionary comprehension here to cleanly shift and correlate in one step
xcorr_data = {
    feat: [wide_df[feat].shift(lag).corr(wide_df[target], min_periods=5) for lag in range(max_lag + 1)]
    for feat in top_15
}

# Build DataFrame: Rows = Features, Columns = Lags
xcorr_matrix = pd.DataFrame(xcorr_data, index=[f"Lag {lag}" for lag in range(max_lag + 1)]).T

# Wrap long feature names for readability
xcorr_matrix.index = [textwrap.fill(str(label), 35) for label in xcorr_matrix.index]

# 3. Set up the plot
plt.figure(figsize=(12, 8), dpi=200)

# Draw the heatmap
sns.heatmap(
    xcorr_matrix,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 6, "alpha": 0.8},
    center=0,
    vmin=-1,
    vmax=1,
    cmap="RdBu_r",
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.75, "aspect": 30, "label": "Pearson Correlation"},
)

plt.title(
    f"Cross-Correlation with Estimated Avoidable Deaths (Up to {max_lag} Days Lag)",
    pad=20,
    fontsize=12,
    fontweight="bold",
)

plt.xticks(rotation=45, ha="right", fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tick_params(left=False, bottom=False)

plt.tight_layout()
plt.show()

## correlation analysis with detrended (rolling-mean adjusted) signals across lags

In this section, we analyse the relationship between the target variable and a set of candidate features in a time series setting.

We compute correlation and cross-correlation across multiple lags, using both raw and detrended signals (via rolling-mean adjustments) to capture short-term dynamics.

We then apply a consistent filtering approach to identify features with potential predictive value at different horizons, and test a small set of feature engineering transformations (e.g. rolling statistics and trends).

In [ ]:
def build_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    data = {}
    ordered_cols = []

    for col in df.columns:
        x = df[col]

        cols = {
            col: x,
            f"{col}_rm7": x.shift(1).rolling(7).mean(),
            f"{col}_std7": x.shift(1).rolling(7).std(),
            f"{col}_trend7": x.shift(1) - x.shift(8),
            f"{col}_sum7": x.shift(1).rolling(7).sum(),
        }

        data.update(cols)
        ordered_cols.extend(cols.keys())

    out = pd.DataFrame(data, index=df.index)
    return out[ordered_cols]

In [ ]:
def analyse_features_relationship(
    df: pd.DataFrame,
    *,
    target_col: str,
    feature_cols: Sequence[str],
    raw_lags: Sequence[int] = (7,),
    dev_lags: Sequence[int] = (1, 3, 7),
    rolling_window: int = 7,
) -> pd.DataFrame:
    target = df[target_col]

    target_roll_mean = target.shift(1).rolling(rolling_window).mean()

    target_dev = target - target_roll_mean

    rows = []

    for feature_col in feature_cols:
        feature = df[feature_col]

        feature_roll_mean = feature.shift(1).rolling(rolling_window).mean()

        feature_dev = feature - feature_roll_mean

        rows.append(
            {
                "feature": feature_col,
                "metric": "dev_corr",
                "lag": 0,
                "value": target_dev.corr(feature_dev),
            }
        )

        rows.extend(
            {
                "feature": feature_col,
                "metric": "dev_corr",
                "lag": lag,
                "value": target_dev.corr(feature_dev.shift(lag)),
            }
            for lag in dev_lags
        )

        rows.append(
            {
                "feature": feature_col,
                "metric": "raw_corr",
                "lag": 0,
                "value": target.corr(feature),
            }
        )

        rows.extend(
            {
                "feature": feature_col,
                "metric": "raw_corr",
                "lag": lag,
                "value": target.corr(feature.shift(lag)),
            }
            for lag in raw_lags
        )

    result = pd.DataFrame(rows)

    table = result.pivot_table(
        index="feature",
        columns=["metric", "lag"],
        values="value",
    ).sort_index(axis=1, level=[0, 1])

    metric_order = ["dev_corr", "raw_corr"]
    lag_order = [0, *dev_lags, *raw_lags]
    lag_order = list(dict.fromkeys(lag_order))

    ordered_cols = [(metric, lag) for metric in metric_order for lag in lag_order if (metric, lag) in table.columns]

    return table[ordered_cols]

In [ ]:
out = build_engineered_features(wide_df)

In [ ]:
table = analyse_features_relationship(out, target_col="estimated_avoidable_deaths", feature_cols=out.columns)
table = table[~table.index.str.startswith("estimated_avoidable_deaths")]

In [ ]:
table[(table[("dev_corr", 1)] > 0.3) | (table[("dev_corr", 1)] < -0.3)].sort_values(
    by=("dev_corr", 1), ascending=False
).round(2)

In [ ]:
table[table.index.isin(["no._of_dtas", "cohorting_&_reverse_queue"])]

## Short-term (D+1) Summary

Short-horizon predictive signal is strongest in **upstream demand** and **system response** metrics.

### Upstream demand (early warning)
- Calls Offered / Answered  
- IUC longest wait (Integrated Urgent Care)  
- IUC queue size  

→ More people trying to access care

---

### System response quality
- Category 2 response time  
- Clinical escalation level  

→ How well the system is coping

---

**Key insight:**  
Increased call volumes, longer IUC wait times, and slower response times are associated with **higher avoidable deaths the following day**.

---

### Hospital pressure
- Escalation Beds Open  

→ Extra beds opened due to pressure  

**Strongest signal overall**, reflecting realised system strain.

---

### Flow / discharge (inverse)
- Complex discharges  
- DTA referrals  

→ Patients leaving the system  

**Inverse relationship:** improved flow reduces pressure and next-day deaths.

In [ ]:
table[(table[("dev_corr", 3)] > 0.25) | (table[("dev_corr", 3)] < -0.25)].sort_values(
    by=("dev_corr", 3), ascending=False
).round(2)

## Long-term (D+3) Summary

Most features show strong **short-term (D+1)** signal but limited predictive power at longer horizons.

A key exception is **Severnside (NHS 111 / triage) activity**, which exhibits a **delayed effect (~3 days)**:
- weak or negative relationship at lag 0  
- positive relationship at lag 3  

**Interpretation:**  
Increased upstream demand initially appears manageable, but translates into higher hospital pressure and avoidable deaths after a delay.

---

### Severnside (triage layer – upstream)

- Calls Received / Answered / Triaged  
- HCP Calls  
- Referrals to ED  

→ Represents **NHS 111 triage activity**, i.e. the earliest point of system demand

---

**Key insight:**  
This reflects a **slow pipeline effect**:
> demand → triage → hospitalisation → outcomes (~3 days)